In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kaggle/kaggle.json


In [6]:
!mkdir -p /root/.kaggle
!cp /kaggle/input/kaggle/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [7]:
!mkdir -p /kaggle/working/kernels_output

!kaggle kernels output jaxa623/0-38357-blend-of-submissions-final-1 -p /kaggle/working/kernels_output/jaxa623_final
!kaggle kernels output hbugrae/blending-submissions-a-cluster-averaging-approach -p /kaggle/working/kernels_output/hbugrae_cluster
!kaggle kernels output antoniyaatanasova/dynamic-cluster-averaging-0-38261 -p /kaggle/working/kernels_output/antoniya_dynamic
!kaggle kernels output analyticaobscura/optimal-fertilizers-eda-playground-0-38265 -p /kaggle/working/kernels_output/analytica_eda
!kaggle kernels output adityaghai01/hill-climb-with-more-ensembles-0-38298 -p /kaggle/working/kernels_output/adityaghai_hillclimb

submission.csv: Skipping, found more recently modified local copy (use --force to force download)
Kernel log downloaded to /kaggle/working/kernels_output/jaxa623_final/0-38357-blend-of-submissions-final-1.log 
Kernel log downloaded to /kaggle/working/kernels_output/hbugrae_cluster/blending-submissions-a-cluster-averaging-approach.log 
submission.csv: Skipping, found more recently modified local copy (use --force to force download)
Kernel log downloaded to /kaggle/working/kernels_output/antoniya_dynamic/dynamic-cluster-averaging-0-38261.log 
submission.csv: Skipping, found more recently modified local copy (use --force to force download)
Kernel log downloaded to /kaggle/working/kernels_output/analytica_eda/optimal-fertilizers-eda-playground-0-38265.log 
submission.csv: Skipping, found more recently modified local copy (use --force to force download)
Kernel log downloaded to /kaggle/working/kernels_output/adityaghai_hillclimb/hill-climb-with-more-ensembles-0-38298.log 


In [10]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

NameError: name 'MAP5SmartEnsemble' is not defined

In [12]:





class MAP5SmartEnsemble:
    def __init__(self, submission_paths, lb_scores, model_names=None):
        """
        MAP@5에 특화된 2단계 앙상블
        
        Args:
            submission_paths: 서브미션 파일 경로 리스트 (5개)
            lb_scores: 각 모델의 LB 점수 리스트 (5개)
            model_names: 모델 이름들 (옵션)
        """
        self.submission_paths = submission_paths
        self.lb_scores = np.array(lb_scores)
        self.model_names = model_names or [f"Model_{i+1}" for i in range(len(submission_paths))]
        self.submissions = []
        self.load_submissions()
        
    def load_submissions(self):
        """서브미션 파일들 로드"""
        for path in self.submission_paths:
            df = pd.read_csv(path)
            self.submissions.append(df)
        print(f"✅ Loaded {len(self.submissions)} submissions")
        print("📊 LB Scores:", dict(zip(self.model_names, self.lb_scores)))
    
    def calculate_model_diversity(self):
        """모델 간 다양성 계산"""
        n_models = len(self.submissions)
        diversity_matrix = np.zeros((n_models, n_models))
        
        print("\n🔍 Calculating model diversity...")
        
        for i in range(n_models):
            for j in range(i+1, n_models):
                # 예측 일치도 계산 (Jaccard similarity의 반대)
                pred_i = self.submissions[i].iloc[:, 1].values  # 첫 번째 컬럼은 ID
                pred_j = self.submissions[j].iloc[:, 1].values
                
                agreement = np.mean(pred_i == pred_j)
                diversity = 1 - agreement  # 다양성 = 1 - 일치도
                
                diversity_matrix[i, j] = diversity
                diversity_matrix[j, i] = diversity
                
                print(f"  {self.model_names[i]} vs {self.model_names[j]}: "
                      f"Agreement={agreement:.3f}, Diversity={diversity:.3f}")
        
        return diversity_matrix
    
    def find_optimal_combination(self, diversity_matrix, min_diversity=0.05):
        """최적 모델 조합 찾기 (2단계: Diversity Filtering)"""
        print(f"\n🎯 Finding optimal combination (min_diversity={min_diversity})...")
        
        n_models = len(self.submissions)
        best_score = -1
        best_combination = None
        
        # 모든 가능한 조합 시도 (3개 이상)
        for r in range(3, n_models + 1):
            for combo in combinations(range(n_models), r):
                # 조합 내 최소 다양성 체크
                min_div = float('inf')
                for i in range(len(combo)):
                    for j in range(i+1, len(combo)):
                        div = diversity_matrix[combo[i], combo[j]]
                        min_div = min(min_div, div)
                
                if min_div >= min_diversity:
                    # 조합의 평균 LB 점수 계산
                    avg_lb = np.mean([self.lb_scores[i] for i in combo])
                    
                    # 다양성 보너스 추가
                    diversity_bonus = min_div * 0.01  # 다양성에 따른 보너스
                    total_score = avg_lb + diversity_bonus
                    
                    if total_score > best_score:
                        best_score = total_score
                        best_combination = combo
                        
                        print(f"  🚀 New best: {[self.model_names[i] for i in combo]}")
                        print(f"     Avg LB: {avg_lb:.5f}, Min Diversity: {min_div:.3f}, "
                              f"Total Score: {total_score:.5f}")
        
        return best_combination
    
    def position_weighted_ensemble(self, selected_models):
        """1단계: Position-Weighted + LB-Confidence Ensemble"""
        print(f"\n🏆 Creating position-weighted ensemble with {len(selected_models)} models...")
        
        # LB 점수 기반 confidence 가중치 계산
        selected_lb_scores = [self.lb_scores[i] for i in selected_models]
        max_lb = max(selected_lb_scores)
        confidence_weights = [score / max_lb for score in selected_lb_scores]
        
        print("📈 Model weights based on LB scores:")
        for i, model_idx in enumerate(selected_models):
            print(f"  {self.model_names[model_idx]}: LB={self.lb_scores[model_idx]:.5f}, "
                  f"Weight={confidence_weights[i]:.3f}")
        
        # 앙상블 예측 계산
        result_df = self.submissions[selected_models[0]].copy()
        n_samples = len(result_df)
        
        final_predictions = []
        
        for idx in range(n_samples):
            # 각 모델의 예측값 수집
            predictions = []
            for model_idx in selected_models:
                pred = self.submissions[model_idx].iloc[idx, 1]  # 첫 번째 컬럼은 ID
                predictions.append(pred)
            
            # Position-weighted voting with confidence
            class_scores = defaultdict(float)
            
            for i, pred in enumerate(predictions):
                # Position weight (모든 예측을 1위로 취급하고 LB confidence만 적용)
                weight = confidence_weights[i]
                class_scores[pred] += weight
            
            # 최고 점수 클래스 선택
            best_class = max(class_scores.items(), key=lambda x: x[1])[0]
            final_predictions.append(best_class)
        
        result_df.iloc[:, 1] = final_predictions
        return result_df
    
    def rank_fusion_ensemble(self, selected_models):
        """대안: Rank Fusion 방식"""
        print(f"\n🔀 Creating rank fusion ensemble...")
        
        selected_lb_scores = [self.lb_scores[i] for i in selected_models]
        weights = np.array(selected_lb_scores) / np.sum(selected_lb_scores)
        
        result_df = self.submissions[selected_models[0]].copy()
        n_samples = len(result_df)
        
        final_predictions = []
        
        for idx in range(n_samples):
            # 각 클래스별 가중 점수 계산
            class_scores = defaultdict(float)
            
            for i, model_idx in enumerate(selected_models):
                pred = self.submissions[model_idx].iloc[idx, 1]
                # LB 점수에 비례한 가중치 적용
                class_scores[pred] += weights[i]
            
            # 최고 점수 클래스 선택
            best_class = max(class_scores.items(), key=lambda x: x[1])[0]
            final_predictions.append(best_class)
        
        result_df.iloc[:, 1] = final_predictions
        return result_df
    
    def create_ensemble(self, method='position_weighted', min_diversity=0.05, save_path='ensemble_submission.csv'):
        """전체 앙상블 파이프라인 실행"""
        print("🚀 Starting MAP@5 Smart Ensemble Pipeline...\n")
        
        # 2단계: Diversity-Aware Model Selection
        diversity_matrix = self.calculate_model_diversity()
        optimal_models = self.find_optimal_combination(diversity_matrix, min_diversity)
        
        if optimal_models is None:
            print(f"⚠️  No combination found with min_diversity={min_diversity}")
            print("📉 Using all models with lower diversity threshold...")
            optimal_models = self.find_optimal_combination(diversity_matrix, min_diversity=0.01)
        
        print(f"\n✅ Selected models: {[self.model_names[i] for i in optimal_models]}")
        
        # 1단계: Smart Ensemble
        if method == 'position_weighted':
            result_df = self.position_weighted_ensemble(optimal_models)
        elif method == 'rank_fusion':
            result_df = self.rank_fusion_ensemble(optimal_models)
        else:
            raise ValueError("Method must be 'position_weighted' or 'rank_fusion'")
        
        # 결과 저장
        result_df.to_csv(save_path, index=False)
        print(f"\n💾 Ensemble saved to: {save_path}")
        
        # 요약 정보 출력
        print(f"\n📋 Ensemble Summary:")
        print(f"  Method: {method}")
        print(f"  Selected Models: {len(optimal_models)}/{len(self.submissions)}")
        print(f"  Expected LB: {np.mean([self.lb_scores[i] for i in optimal_models]):.5f}")
        
        return result_df

# 사용 예제
if __name__ == "__main__":
    # 예제 데이터
    submission_paths = [
        '/kaggle/working/kernels_output/adityaghai_hillclimb/submission.csv',  # 여기에 실제 파일 경로들을 넣으세요
        '/kaggle/working/kernels_output/analytica_eda/submission.csv',
        '/kaggle/working/kernels_output/antoniya_dynamic/submission.csv', 
        '/kaggle/working/kernels_output/jaxa623_final/submission.csv'
    ]
    
    lb_scores = [0.38357, 0.38213, 0.38261, 0.38298]  # 실제 LB 점수들
    model_names = ['adityaghai', 'analyritica', 'anatoniya_dynamic', 'jaxa623']
    
    # 앙상블 생성
    ensemble = MAP5SmartEnsemble(submission_paths, lb_scores, model_names)
    
    # 방법 1: Position-Weighted Ensemble
    result1 = ensemble.create_ensemble(
        method='position_weighted',
        min_diversity=0.05,
        save_path='smart_ensemble_position.csv'
    )
    
    # 방법 2: Rank Fusion Ensemble  
    result2 = ensemble.create_ensemble(
        method='rank_fusion',
        min_diversity=0.05,
        save_path='smart_ensemble_rank.csv'
    )

✅ Loaded 4 submissions
📊 LB Scores: {'adityaghai': 0.38357, 'analyritica': 0.38213, 'anatoniya_dynamic': 0.38261, 'jaxa623': 0.38298}
🚀 Starting MAP@5 Smart Ensemble Pipeline...


🔍 Calculating model diversity...
  adityaghai vs analyritica: Agreement=0.692, Diversity=0.308
  adityaghai vs anatoniya_dynamic: Agreement=0.516, Diversity=0.484
  adityaghai vs jaxa623: Agreement=0.829, Diversity=0.171
  analyritica vs anatoniya_dynamic: Agreement=0.747, Diversity=0.253
  analyritica vs jaxa623: Agreement=0.803, Diversity=0.197
  anatoniya_dynamic vs jaxa623: Agreement=0.652, Diversity=0.348

🎯 Finding optimal combination (min_diversity=0.05)...
  🚀 New best: ['adityaghai', 'analyritica', 'anatoniya_dynamic']
     Avg LB: 0.38277, Min Diversity: 0.253, Total Score: 0.38530

✅ Selected models: ['adityaghai', 'analyritica', 'anatoniya_dynamic']

🏆 Creating position-weighted ensemble with 3 models...
📈 Model weights based on LB scores:
  adityaghai: LB=0.38357, Weight=1.000
  analyritica: LB=0

In [ ]:
# 1. 파일 경로와 LB 점수 설정
submission_paths = ['path1.csv', 'path2.csv', ...]
lb_scores = [0.38345, 0.38298, 0.38285, 0.38265, 0.38213]

# 2. 앙상블 객체 생성
ensemble = MAP5SmartEnsemble(submission_paths, lb_scores)

# 3. 앙상블 실행
result = ensemble.create_ensemble(
    method='position_weighted',  # 또는 'rank_fusion'
    min_diversity=0.05,         # 최소 다양성 임계값
    save_path='final_ensemble.csv'
)